# Part 4 — Test frozen strategies

**Input from Part 3:** `artifacts/strategy/frozen_strategy.json`  
Features: full-timeline `prepare_train_val_test()` · Model: existing XGBoost (no retrain)

**Method:**
1. Strategy optimized on validation at **FP=$30**
2. Frozen and scored once on the untouched test set
3. Robustness: same frozen predictions rescored under `$10, $30, $50, $100` (no re-tune)

Previous: [`03_optimize_thresholds.ipynb`](03_optimize_thresholds.ipynb)


In [ ]:
from pathlib import Path
import sys

# Project root (works when cwd is notebooks/ or project root)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import (
    DATA_DIR,
    MODELS_DIR,
    ARTIFACTS_DIR,
    STRATEGY_DIR,
    TRAIN_CSV,
    TEST_CSV,
    XGB_MODEL_PATH,
    LOGISTIC_MODEL_PATH,
    FROZEN_STRATEGY_PATH,
)

print("Project root:", ROOT)
print("Data dir:", DATA_DIR)
print("Models dir:", MODELS_DIR)


## Section 4 - Testing frozen stategies on the Test set:


In [ ]:
xgb_test_prob = best_xgb.predict_proba(X_test)[:, 1]


In [ ]:

row30 = best_high_recall_by_fp_cost.loc[
    best_high_recall_by_fp_cost["fp_cost"] == 30
].iloc[0]

test_strategies_30 = build_rule_strategies(
    df_test,
    amt=row30["amt_threshold"],
    minutes=row30["minutes_threshold"],
    tx_23=row30["transactions_23_threshold"],
    tx_60=row30["transactions_60min_threshold"],
)

rules_high_precision_test = test_strategies_30["high_precision"]
rules_high_recall_test = high_recall_masks_test[30]
rules_only_test_predictions = best_rule_masks_test[30]


In [ ]:
# Frozen strategies are selected on validation for each FP cost, then scored once on test
test_results_by_fp = []

for fp_cost in fp_costs:
    rules_test_pred = best_rule_masks_test[fp_cost]

    xgb_thr = float(
        best_xgb_threshold_by_fp.loc[
            best_xgb_threshold_by_fp["fp_cost"] == fp_cost, "threshold"
        ].iloc[0]
    )
    xgb_test_pred = xgb_test_prob >= xgb_thr

    hybrid_row = best_hybrid_3_by_fp.loc[
        best_hybrid_3_by_fp["fp_cost"] == fp_cost
    ].iloc[0]
    high_thr = float(hybrid_row["high_threshold"])
    low_thr = float(hybrid_row["low_threshold"])
    hybrid_test_pred = (
        (xgb_test_prob >= high_thr) |
        (
            (xgb_test_prob >= low_thr) &
            high_recall_masks_test[fp_cost].to_numpy()
        )
    )

    rules_res = evaluate_predictions(
        y_true=y_test,
        predictions=rules_test_pred,
        amounts=df_test["amt"],
        fp_cost=fp_cost,
    )
    rules_res.update({"strategy": "Rules only", "fp_cost": fp_cost})

    xgb_res = evaluate_predictions(
        y_true=y_test,
        predictions=xgb_test_pred,
        amounts=df_test["amt"],
        fp_cost=fp_cost,
    )
    xgb_res.update({
        "strategy": "XGBoost only",
        "fp_cost": fp_cost,
        "threshold": xgb_thr,
    })

    hybrid_res = evaluate_predictions(
        y_true=y_test,
        predictions=hybrid_test_pred,
        amounts=df_test["amt"],
        fp_cost=fp_cost,
    )
    hybrid_res.update({
        "strategy": "Hybrid",
        "fp_cost": fp_cost,
        "high_threshold": high_thr,
        "low_threshold": low_thr,
    })

    test_results_by_fp.extend([rules_res, xgb_res, hybrid_res])

test_comparison_by_fp_df = pd.DataFrame(test_results_by_fp)
test_comparison_by_fp_df


In [ ]:

xgb_test_predictions = (
    xgb_test_prob >= float(
        best_xgb_threshold_by_fp.loc[
            best_xgb_threshold_by_fp["fp_cost"] == 30, "threshold"
        ].iloc[0]
    )
)

hybrid_best_predictions = (
    (xgb_test_prob >= float(best_hybrid_3["high_threshold"])) |
    (
        (xgb_test_prob >= float(best_hybrid_3["low_threshold"])) &
        rules_high_recall_test.to_numpy()
    )
)


In [ ]:
rules_test_result = evaluate_predictions(
    y_true=y_test,
    predictions=rules_only_test_predictions,
    amounts=df_test["amt"],
    fp_cost=30,
)

xgb_test_result = evaluate_predictions(
    y_true=y_test,
    predictions=xgb_test_predictions,
    amounts=df_test["amt"],
    fp_cost=30,
)

hybrid_test_result = evaluate_predictions(
    y_true=y_test,
    predictions=hybrid_best_predictions,
    amounts=df_test["amt"],
    fp_cost=30,
)


In [ ]:
rules_test_result["strategy"] = "Rules only"

xgb_test_result["strategy"] = "XGBoost only"
xgb_test_result["threshold"] = float(
    best_xgb_threshold_by_fp.loc[
        best_xgb_threshold_by_fp["fp_cost"] == 30, "threshold"
    ].iloc[0]
)

hybrid_test_result["strategy"] = "Hybrid"
hybrid_test_result["high_threshold"] = float(best_hybrid_3["high_threshold"])
hybrid_test_result["low_threshold"] = float(best_hybrid_3["low_threshold"])


In [ ]:
# Robustness results after full-timeline features + optimize@$30
fp_cost_sensitivity_df = pd.read_csv(ARTIFACTS_DIR / "fp_cost_sensitivity.csv")
test_comparison_by_fp_df = pd.read_csv(ARTIFACTS_DIR / "test_results_by_fp_cost.csv")

mean_pct_reduction = fp_cost_sensitivity_df["hybrid_vs_xgb_pct_reduction"].mean()
median_pct_reduction = fp_cost_sensitivity_df["hybrid_vs_xgb_pct_reduction"].median()
min_pct_reduction = fp_cost_sensitivity_df["hybrid_vs_xgb_pct_reduction"].min()
max_pct_reduction = fp_cost_sensitivity_df["hybrid_vs_xgb_pct_reduction"].max()
std_pct_reduction = fp_cost_sensitivity_df["hybrid_vs_xgb_pct_reduction"].std(ddof=0)
hybrid_wins = (fp_cost_sensitivity_df["winner"] == "Hybrid").sum()
baseline_30 = float(
    fp_cost_sensitivity_df.loc[
        fp_cost_sensitivity_df["fp_cost"] == 30, "hybrid_vs_xgb_pct_reduction"
    ].iloc[0]
)

print("Robustness table (frozen @$30 strategy, varying FP cost on test):")
display(fp_cost_sensitivity_df)
display(test_comparison_by_fp_df)

print(
    f"$30 baseline hybrid vs XGBoost reduction: {baseline_30:.2f}%\n"
    f"Mean across robustness grid: {mean_pct_reduction:.2f}%\n"
    f"Median: {median_pct_reduction:.2f}% | Min: {min_pct_reduction:.2f}% | "
    f"Max: {max_pct_reduction:.2f}% | Std: {std_pct_reduction:.2f}%\n"
    f"Hybrid wins: {hybrid_wins} / {len(fp_cost_sensitivity_df)} FP costs"
)

test_comparison_df = (
    test_comparison_by_fp_df[test_comparison_by_fp_df["fp_cost"] == 30]
    .sort_values("total_cost")
    .reset_index(drop=True)
)
print("\n$30 baseline comparison:")
display(test_comparison_df)


## Final Test-Set Results

**Method:** full-timeline features → optimize on validation at **FP=$30** → freeze → test once → robustness under `$10, $30, $50, $100` without re-tuning.

### Frozen @$30 strategy

- **XGBoost threshold:** `0.082`
- **Hybrid:** `(prob >= 0.092) OR ((prob >= 0.016) AND high-recall rules)`
- Rules: `high_recall`, amt>=`250`, minutes<=`50`, tx23>=`2`, tx60>=`2`

### Robustness table (same frozen predictions, varying FP cost)

| FP cost | Hybrid | XGBoost only | Rules only | Hybrid savings vs XGB | Hybrid % reduction vs XGB | Winner |
|---:|---:|---:|---:|---:|---:|:---|
| $10 | $66,404.48 | $92,041.04 | $318,045.56 | $25,636.56 | 27.85% | Hybrid |
| $30 | $104,504.48 | $116,141.04 | $408,025.56 | $11,636.56 | 10.02% | Hybrid |
| $50 | $142,604.48 | $140,241.04 | $498,005.56 | $-2,363.44 | -1.69% | XGBoost only |
| $100 | $237,854.48 | $200,491.04 | $722,955.56 | $-37,363.44 | -18.64% | XGBoost only |

### Distribution of hybrid improvement vs XGBoost

- **At $30 (optimization baseline):** 10.02%
- **Mean across robustness grid:** 4.39%
- **Median:** 4.17%
- **Min / Max:** -18.64% / 27.85%
- **Std. dev.:** 16.95%
- **Hybrid wins:** 2 / 4 FP costs

### $30 baseline highlight

| Strategy | Total Cost | Precision | Recall |
|:---|---:|---:|---:|
| Hybrid | $104,504.48 | 48.80% | 84.66% |
| XGBoost only | $116,141.04 | 59.81% | 83.59% |
| Rules only | $408,025.56 | 20.97% | 55.66% |


## Project Summary

This project compares rules-only, XGBoost, and a hybrid fraud strategy.

### Data / features

Past-looking features are built on the **full chronological timeline** (`prepare_train_val_test`), then split into train/val/test. Train-row features stay the same as before (test is after train), so the existing XGBoost model was **not retrained**. Val/test now retain earlier card history.

### Decision process

1. Optimize rules + XGBoost + hybrid thresholds on validation at **FP=$30**
2. Freeze that strategy
3. Evaluate once on the untouched test set
4. Robustness: rescore the same frozen test predictions under `$10, $30, $50, $100`

### Frozen @$30 thresholds

- XGBoost: `0.082`
- Hybrid: high `0.092`, low `0.016`

## Final Test Performance (robustness)

| FP cost | Hybrid | XGBoost only | Rules only | Hybrid savings vs XGB | Hybrid % reduction vs XGB | Winner |
|---:|---:|---:|---:|---:|---:|:---|
| $10 | $66,404.48 | $92,041.04 | $318,045.56 | $25,636.56 | 27.85% | Hybrid |
| $30 | $104,504.48 | $116,141.04 | $408,025.56 | $11,636.56 | 10.02% | Hybrid |
| $50 | $142,604.48 | $140,241.04 | $498,005.56 | $-2,363.44 | -1.69% | XGBoost only |
| $100 | $237,854.48 | $200,491.04 | $722,955.56 | $-37,363.44 | -18.64% | XGBoost only |

At **$30**, hybrid vs XGBoost test-cost reduction is **10.0%**.

Across the robustness grid: mean **4.4%**, range **-18.6% to 27.9%**, hybrid wins **2/4**.

## Final Conclusion

Tune for **$30**, then check robustness. Hybrid helps at lower FP costs; at higher FP costs the frozen hybrid can lose to XGBoost-only because it was not re-tuned for those costs.

## Limitations

Results depend on the $30 optimization assumption, label timing for previous-fraud features, and using transaction amount as false-negative cost.
